# 단일 클래스 알약 탐지기 학습 — 탐지기 데이터 v2 (Colab GPU)

레포 공식 노트북 `notebooks/baseline/train_detector.ipynb`(main)를 **그대로** 실행한다. 설정 셀 하나만 이번 실행 값으로 바꾼다.

**왜 이렇게 실행하나:** Colab 노트북 커널은 Python 3.13이고, 레포는 Python 3.12만 지원한다(`requires-python <3.13`).
그래서 이 노트북 커널에서 학습 코드를 직접 불러올 수 없다. 대신 공식 노트북을 **레포 환경(`uv run`, Python 3.12, 고정 패키지)** 안에서 실행하고,
실행이 끝난 노트북(그래프·출력 포함)을 드라이브 결과 폴더에 저장한다.

- 데이터: 팀 Drive 탐지기 v2 `gitkk_archive/detector_data/1. selected_data/v2/` (train 8,143장 / val 2,026장, PR #34)
- 설정: 2차 제출 탐지기와 같음 (SSDLite320, batch 8, lr 0.005, 최대 50 epoch, early stopping 7, 증강 없음, seed 42). 데이터만 v2
- 실행 이름·tag·저장 위치: 레포 `docs/experiments/run_conventions.md`
- 결과: 드라이브 `MyDrive/gitkk_runs/detector/<run_name>/` → 로컬 `outputs/detector/<run_name>/` (폴더 단위)

```python
run_name = "20261104-ssdlite320-det-v2-baseline-01"
# tags = [
#     "task:detector",
#     "stage:baseline",
#     "model:ssdlite320",
#     "data:det-v2",
#     "change:baseline",
#     "device:cuda",
# ]
```

**실행:** VS Code에서 커널을 Colab **GPU** 서버에 연결하고 0-1번부터 순서대로 실행한다.
- 학습(2번)은 **백그라운드**로 돌고 셀은 바로 끝난다. 진행 상황은 3번 셀을 여러 번 실행해서 본다.
- 중단 후 재개: 2번 셀을 다시 실행하면 드라이브의 `last.pt`에서 이어서 학습한다 (같은 run_name).
- 처음부터 다시 학습할 때는 run_name 끝 순번을 올린다 (`-01` → `-02`, 1번 셀).
- 각 셀은 문제가 있으면 그 자리에서 멈춘다. **처음 멈춘 셀**의 메시지를 확인한다.

## 0. Colab 준비

GPU 확인 → 레포(main)와 의존성 설치(README의 Colab 안내) → 드라이브 연결 → 탐지기 v2 복사·검사.

In [ ]:
# 0-1. GPU 확인
!nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
# 0-2. 레포(main)와 의존성 (README의 Colab 안내와 같은 방식, uv 버전은 pyproject 고정값)
%cd /content
!test -d gitkk-pill-detection || git clone https://github.com/auranova1125/gitkk-pill-detection.git
%cd /content/gitkk-pill-detection
!git checkout main && git pull --ff-only
!git log --oneline -1
%pip install -q uv==0.13.4
!uv sync --locked
!uv run python -c 'import torch; print(f"cuda={torch.cuda.is_available()}")'


In [ ]:
# 0-3. 드라이브 연결
import os
if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')  # 로그인 창이 뜨면 드라이브 접근을 허용한다
assert os.path.isdir('/content/drive/MyDrive'), (
    "드라이브 연결 실패. VS Code 명령 팔레트(Cmd+Shift+P)에서 "
    "'Colab: Mount Google Drive to Server...'를 실행하고, 노트북 끝에 추가된 셀을 실행한 뒤 이 셀을 다시 실행한다"
)
print("드라이브 연결됨")

In [ ]:
# 0-4. Drive의 탐지기 v2 폴더 찾기
import glob, os, unicodedata
from pathlib import Path

PKG = "/content/detector_v2"



def nfc(text):
    return unicodedata.normalize("NFC", text)

patterns = [
    "/content/drive/MyDrive/gitkk_archive",
    "/content/drive/MyDrive/*/gitkk_archive",
    "/content/drive/Shareddrives/*/gitkk_archive",
    "/content/drive/Shareddrives/*/*/gitkk_archive",
    "/content/drive/.shortcut-targets-by-id/*/gitkk_archive",
    "/content/drive/.shortcut-targets-by-id/*/*/gitkk_archive",
]
archives = [p for pat in patterns for p in glob.glob(pat) if os.path.isdir(os.path.join(p, "detector_data"))]
print("찾은 gitkk_archive:", archives)
assert archives, (
    "gitkk_archive를 찾지 못했다. 드라이브 웹에서 '공유 문서함'의 gitkk_archive를 "
    "우클릭 > 정리 > 바로가기 추가 > '내 드라이브'에 둔 뒤 1번 셀부터 다시 실행한다"
)
det = Path(archives[0]) / "detector_data"
selected = [d for d in det.iterdir() if d.is_dir() and "selected_data" in nfc(d.name)]
print("detector_data 안:", sorted(nfc(d.name) for d in det.iterdir()))
assert selected, "detector_data 안에 'selected_data' 폴더가 없다. 위 목록을 확인한다"
V2 = selected[0] / "v2"
assert V2.is_dir(), f"{V2} 가 없다"
print("v2 폴더:", V2)
print("v2 안:", sorted(nfc(p.name) for p in V2.iterdir()))
for need in ["annotations/train.json", "annotations/validation.json", "split_manifest.csv", "data_settings.json"]:
    assert (V2 / need).exists(), f"v2에 {need} 가 없다. 위 목록을 확인한다"


### 0-5. v2 복사와 검사

Drive v2를 `/content/detector_v2`로 복사한다(사진 약 1만 장, **수십 분**). 32개씩 동시에 받고, 이미 받은 파일(크기가 같은 파일)은 건너뛴다.
연결이 끊기면 0-1~0-4를 다시 실행한 뒤 이 셀을 다시 실행하면 이어서 받는다(런타임이 초기화됐으면 처음부터).
경로 설정(`data_settings.json`)은 Colab 위치에 맞게 새로 쓴다. 마지막 줄이 `passed <hash> 8143 2026`이면 정상이다.

In [ ]:
# 0-5. 복사 → 경로 설정 → 데이터 검사
# 사진을 32개씩 동시에 복사한다. 크기가 같은 파일은 건너뛰므로, 끊겼으면 이 셀만 다시 실행하면 이어서 받는다.
import csv, json, os, shutil, subprocess
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

jobs = []
for dirpath, _, files in os.walk(V2):
    rel = os.path.relpath(dirpath, V2)
    os.makedirs(os.path.join(PKG, rel), exist_ok=True)
    for name in files:
        src, dst = os.path.join(dirpath, name), os.path.join(PKG, rel, name)
        if name == "data_settings.json" and rel == ".":
            continue  # 아래에서 Colab 경로로 새로 쓴다
        if not os.path.exists(dst) or os.path.getsize(dst) != os.path.getsize(src):
            jobs.append((src, dst))
print("복사할 파일:", len(jobs))
with ThreadPoolExecutor(32) as ex:
    list(tqdm(ex.map(lambda j: shutil.copyfile(*j), jobs), total=len(jobs), desc="복사"))

rows = list(csv.DictReader(open(f"{PKG}/split_manifest.csv", encoding="utf-8-sig")))
roots = {}
for dataset in sorted({r["dataset"] for r in rows}):
    sample = next(r for r in rows if r["dataset"] == dataset)["image_path"]
    cands = [Path(PKG), Path(PKG) / dataset, Path(PKG) / "images", Path(PKG) / dataset / "images"]
    hit = next((c for c in cands if (c / sample).is_file()), None)
    assert hit is not None, f"{dataset} 사진을 찾지 못했다: {sample}. v2 안 폴더 구조를 확인한다"
    roots[dataset] = str(hit)
settings = {"package_dir": PKG, "image_roots": roots}
json.dump(settings, open(f"{PKG}/data_settings.json", "w"), indent=2)
print("새 설정:", settings)

%cd /content/gitkk-pill-detection
check = subprocess.run(
    ["uv", "run", "python", "main.py", "check-detector-data",
     "--settings", f"{PKG}/data_settings.json", "--report", "/content/check_v2.json"],
    capture_output=True, text=True,
)
print(check.stdout[-2000:], check.stderr[-2000:])
report = json.load(open("/content/check_v2.json"))
assert report["status"] == "passed", report.get("error")
summary = report["data_summary"]
print(report["status"], summary["split_hash"][:8], summary["num_train_images"], summary["num_val_images"])
if (summary["num_train_images"], summary["num_val_images"]) != (8143, 2026):
    print("주의: 보고서(8,143 / 2,026)와 장수가 다르다. 팀원과 v2 버전을 확인한다")


## 1. 공식 노트북 준비

서버에 받은 레포의 `notebooks/baseline/train_detector.ipynb`를 읽어 **설정 셀만** 아래 값으로 바꾼 실행용 사본을 만든다. 나머지 셀은 공식 노트북 그대로다.

In [ ]:
# 1. 공식 노트북 + 이번 실행 설정 → 실행용 사본
import json
from pathlib import Path
REPO = Path("/content/gitkk-pill-detection")
SETTINGS_CELL = "run_name = \"20261104-ssdlite320-det-v2-baseline-01\"\ndevice = \"auto\"\nshow_progress = True\n# tags = [\n#     \"task:detector\",\n#     \"stage:baseline\",\n#     \"model:ssdlite320\",\n#     \"data:det-v2\",\n#     \"change:baseline\",\n#     \"device:cuda\",\n# ]\n\n# Colab: 결과를 유지하도록 output_dir을 드라이브로 둔다 (공식 노트북 안내)\noutput_dir = Path(\"/content/drive/MyDrive/gitkk_runs/detector\")\ndata_settings_path = Path(\"/content/detector_v2/data_settings.json\")  # 0-5번에서 만든 설정\ndata_factory_path = \"data_loading.detector:build_detector_inputs\"\n\n# 2차 제출 탐지기와 같은 학습 설정 (공식 노트북 smoke 값에서 변경: batch 2→8, epoch 2→50, pretrained False→True)\nbatch_size = 8\nmax_epochs = 50\npretrained_backbone = True\n\n# 같은 run_name에 last.pt가 있으면 이어서 학습한다 (가이드: 재개는 같은 run_name).\n# last.pt 없이 best.pt/metrics.csv만 있으면 학습 코드가 덮어쓰기를 막고 멈춘다 → run_name 순번을 올린다\n_last = output_dir / run_name / \"last.pt\"\nresume_from = _last if _last.exists() else None\n\nconfig = DetectorConfig(\n    run_name=run_name,\n    output_dir=output_dir,\n    learning_rate=5e-3,\n    momentum=0.9,\n    weight_decay=5e-4,\n    score_threshold=0.25,\n    nms_threshold=0.45,\n    early_stopping_patience=7,\n    data_factory_path=data_factory_path,\n    data_settings_path=data_settings_path,\n    pretrained_backbone=pretrained_backbone,\n    device=device,\n    seed=42,\n    batch_size=batch_size,\n    max_epochs=max_epochs,\n    resume_from=resume_from,\n    show_progress=show_progress,\n)\nrun_dir = config.run_dir\nprint(f\"====================\\nrun_name: {run_name}\\n====================\\n\")\nprint(f\"device: {config.device}\\nmodel: {config.model_name}\\nrun_dir: {run_dir}\")\nconfig_view = config.to_dict()\nfor key in [\"batch_size\", \"max_epochs\", \"early_stopping_patience\", \"horizontal_flip\", \"vertical_flip\", \"brightness\", \"contrast\"]:\n    print(f\"{key}: {config_view[key]}\")\nprint(f\"resume_from: {config.resume_from}\")\n"
official = json.loads((REPO / "notebooks/baseline/train_detector.ipynb").read_text())
idx = [i for i, c in enumerate(official["cells"]) if c["cell_type"] == "code" and "".join(c["source"]).lstrip().startswith("run_name =")]
assert len(idx) == 1, f"공식 노트북에서 설정 셀을 하나만 찾아야 한다: {idx}"
official["cells"][idx[0]]["source"] = SETTINGS_CELL.splitlines(True)
RUN_NAME = "20261104-ssdlite320-det-v2-baseline-01"
RUN_NB = REPO / "notebooks/baseline" / f"_colab_{RUN_NAME}.ipynb"   # project_root를 찾도록 레포 안에 둔다 (Git에 올리지 않음)
RUN_NB.write_text(json.dumps(official, ensure_ascii=False, indent=1))
RUN_DIR = Path("/content/drive/MyDrive/gitkk_runs/detector") / RUN_NAME
LOG = Path("/content") / f"{RUN_NAME}.log"
print("실행용 노트북:", RUN_NB)
print("바뀐 설정 셀:\n" + SETTINGS_CELL)


## 2. 학습 실행 (백그라운드)

레포 환경(Python 3.12)에서 공식 노트북을 처음부터 끝까지 실행한다. 학습·평가·시각화까지 모두 포함된다.
끝나면 출력이 담긴 노트북이 `RUN_DIR/train_detector_executed.ipynb`로 저장된다.

In [ ]:
# 2. 공식 노트북을 레포 환경에서 실행 (백그라운드)
import subprocess, os, json
assert os.path.isfile("/content/detector_v2/data_settings.json"), "v2가 준비돼 있지 않다. 0-4, 0-5번 셀을 실행한다"
running = subprocess.run(["pgrep", "-f", "nbconvert.*" + RUN_NB.name], capture_output=True, text=True).stdout.strip()
assert not running, f"이미 실행 중이다 (pid {running}). 3번 셀로 진행 상황을 본다"

# 레포 .venv(Python 3.12)를 가리키는 커널을 등록한다.
# Colab의 /etc/ipython/ipython_config.py가 모든 커널에 google.colab 커널 클래스·확장을 지정하는데,
# 레포 환경에는 google.colab이 없어 커널이 바로 죽는다 → 커널 실행 인자로 그 설정을 덮어쓴다
subprocess.run(["uv", "run", "python", "-m", "ipykernel", "install", "--user", "--name", "gitkk-uv"],
               cwd=str(REPO), check=True)
OVERRIDES = [
    "--IPKernelApp.kernel_class=ipykernel.ipkernel.IPythonKernel",
    "--InteractiveShellApp.extensions=[]",
    "--InteractiveShellApp.exec_lines=[]",
    "--InteractiveShellApp.reraise_ipython_extension_failures=False",
]
spec_path = "/root/.local/share/jupyter/kernels/gitkk-uv/kernel.json"
spec = json.load(open(spec_path))
spec["argv"] = [a for a in spec["argv"] if a not in OVERRIDES] + OVERRIDES
json.dump(spec, open(spec_path, "w"), indent=1)

# 커널이 실제로 뜨는지 15초 시험 (124 = 15초 동안 살아 있었다 = 정상)
test = [a.replace("{connection_file}", "/tmp/gitkk_test_kernel.json") for a in spec["argv"]]
r = subprocess.run(["timeout", "15"] + test, cwd=str(REPO), capture_output=True, text=True)
assert r.returncode == 124, "커널 시험 실패 (exit %d):\n%s" % (r.returncode, r.stderr[-3000:])
print("커널 시험 통과")

RUN_DIR.mkdir(parents=True, exist_ok=True)
args = [
    "nohup", "uv", "run", "--with", "nbconvert",
    "jupyter", "nbconvert", "--to", "notebook", "--execute",
    "--ExecutePreprocessor.timeout=-1", "--ExecutePreprocessor.kernel_name=gitkk-uv",
    "--output-dir", str(RUN_DIR), "--output", "train_detector_executed.ipynb", str(RUN_NB),
]
subprocess.Popen(args, cwd=str(REPO), stdout=open(LOG, "w"), stderr=subprocess.STDOUT, start_new_session=True)
print("시작했다. 로그:", LOG)


## 3. 진행 상황 확인 (여러 번 실행)

로그 끝부분과, 드라이브에 쌓이는 epoch별 지표(`metrics.csv`)를 보여 준다. `map_75_95`는 약 이름을 무시한 탐지 점수다.

In [ ]:
# 3. 진행 상황
import subprocess, pandas as pd
alive = subprocess.run(["pgrep", "-f", f"nbconvert.*{RUN_NB.name}"], capture_output=True, text=True).stdout.strip()
print("상태:", "실행 중" if alive else "끝남(또는 시작 안 함)")
print("----- 로그 끝부분 -----")
print(LOG.read_text()[-3000:] if LOG.exists() else "(로그 없음)")
m = RUN_DIR / RUN_NAME / "metrics.csv"
m = m if m.exists() else RUN_DIR / "metrics.csv"
if m.exists():
    df = pd.read_csv(m)
    best = df.loc[df["map_75_95"].idxmax()]
    print(f"----- epochs={len(df)}  best epoch={int(best.epoch)}  map50={best.map_50:.4f}  map75_95={best.map_75_95:.4f}")
    print(df[["epoch", "train_loss", "map_50", "map_75_95"]].tail(5).to_string(index=False))
else:
    print("(아직 metrics.csv 없음)")
executed = RUN_DIR / "train_detector_executed.ipynb"
if not alive and executed.exists():
    print("완료: 출력이 담긴 노트북 →", executed)


## 4. 로컬로 가져갈 파일

드라이브 `MyDrive/gitkk_runs/detector/20261104-ssdlite320-det-v2-baseline-01/` **폴더 전체**를 로컬 `outputs/detector/20261104-ssdlite320-det-v2-baseline-01/`에 둔다
(checkpoint, `label_map.json`, `resolved_config.json`, `metrics.csv`, 실행된 노트북 `train_detector_executed.ipynb`).

그다음 v2 val로 탐지+분류(e2e) 채점을 한다. 통합 추론 결과는 `outputs/inference/20261104-ssdlite320-resnet50-det-v2-evaluation-01/`에 저장한다.